# 🌿 AgroCure v4 — Plant Disease Classification
**Architecture:** Hierarchical two-stage model (Stage 1: plant → Stage 2: disease)  
**Backbone:** EfficientNetV2-S  
**Classes:** 30 diseases across 16 plants  
**Dataset:** AgroCure v4 (7,057 images, augmented to 200/class during training)

---
### Notebook Flow
| Cell | What | Time |
|------|------|------|
| 1 | Install dependencies | 2 min |
| 2 | DagsHub / MLflow setup | 1 min |
| 3 | Copy dataset + scripts | 7 min |
| 4 | Merge similar classes | 1 min |
| 5 | Augmentation (→200/class) | 18 min |
| 6 | Train/val/test splits | 1 min |
| 7 | Train Stage 1 (plant classifier) | 70 min |
| 8 | Evaluate Stage 1 | 4 min |
| 9A-9F | Train Stage 2 (per-plant disease) | 3 hrs |
| 10 | Temperature calibration | 8 min |
| 11 | Full pipeline evaluation | 10 min |
| 12 | Post-training EDA | 15 min |
| 13 | Upload metrics to DagsHub | 1 min |
| 14 | Save & backup everything | 5 min |

In [ ]:
# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
%pip install -q dagshub mlflow timm scikit-learn matplotlib seaborn pandas pillow imagehash

In [ ]:
# ============================================================
# CELL 2 — DagsHub / MLflow Setup
# ============================================================
# Credentials come from environment variables — never hardcode them.
# Set before running, e.g. in Kaggle "Secrets" or:
#   os.environ["DAGSHUB_USERNAME"] = "..."   # your DagsHub username
#   os.environ["DAGSHUB_TOKEN"]    = "..."   # your DagsHub access token
import os, dagshub, mlflow

os.environ["MLFLOW_TRACKING_USERNAME"] = os.environ.get("DAGSHUB_USERNAME", "")
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ.get("DAGSHUB_TOKEN", "")

dagshub.init(repo_owner='Faira73', repo_name='Agro-Mind', mlflow=True)
mlflow.set_experiment("AgroCure-v4")
print("MLflow connected ✓")
print("Dashboard → https://dagshub.com/Faira73/Agro-Mind.mlflow")

In [ ]:
# ============================================================
# CELL 3 — Setup: Copy Dataset + Scripts
# ============================================================
import os, shutil

WORK_DIR    = "/kaggle/working/agrocure_v4"
DATASET_SRC = "/kaggle/input/datasets/khaled02/agrocure-v4-dataset/dataset"
SCRIPTS_SRC = "/kaggle/input/datasets/khaled02/agrocure-scripts-v4/agrocure_v4"

os.makedirs(WORK_DIR, exist_ok=True)

# Copy scripts
for f in os.listdir(SCRIPTS_SRC):
    src = os.path.join(SCRIPTS_SRC, f)
    dst = os.path.join(WORK_DIR, f)
    if os.path.isfile(src):
        shutil.copy(src, dst)
print("Scripts copied ✓")

# Copy dataset
DATASET_DST = os.path.join(WORK_DIR, "dataset")
if not os.path.exists(DATASET_DST):
    print("Copying dataset (few minutes)...")
    shutil.copytree(DATASET_SRC, DATASET_DST)
    print("Dataset copied ✓")
else:
    print("Dataset already exists ✓")

classes = os.listdir(DATASET_DST)
print(f"Classes found: {len(classes)}")
print("Setup complete ✓")

In [ ]:
# ============================================================
# CELL 4 — Merge Similar Classes
# ============================================================
import runpy, os, sys

sys.path.insert(0, "/kaggle/working/agrocure_v4")
os.chdir("/kaggle/working/agrocure_v4")
runpy.run_path("01_merge_classes.py", run_name="__main__")

In [ ]:
# ============================================================
# CELL 5 — Augmentation (target: 200 images per class)
# ============================================================
import runpy, os
os.chdir("/kaggle/working/agrocure_v4")
runpy.run_path("02_augment_v4.py", run_name="__main__")

In [ ]:
# ============================================================
# CELL 6 — Make Splits (full + per-plant)
# ============================================================
import runpy, os
os.chdir("/kaggle/working/agrocure_v4")
runpy.run_path("03_make_splits_v4.py", run_name="__main__")

---
## 🌱 Stage 1 — Plant Classifier
Trains one model to identify **which plant** is in the image (16 plants).  
Expected accuracy: ~97–98%

In [ ]:
# ============================================================
# CELL 7 — Train Stage 1: Plant Classifier
# ============================================================
import runpy, os, mlflow
mlflow.end_run()
os.chdir("/kaggle/working/agrocure_v4")
runpy.run_path("04_train_stage1.py", run_name="__main__")

In [ ]:
# ============================================================
# CELL 8 — Evaluate Stage 1 Only
# ============================================================
import os, json, csv, torch
import torch.nn as nn
import numpy as np
from torchvision import transforms, models
from PIL import Image
import sys
sys.path.insert(0, "/kaggle/working/agrocure_v4")
import config_v4 as config

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN   = [0.485, 0.456, 0.406]
STD    = [0.229, 0.224, 0.225]

with open(config.STAGE1_CLASSES) as f:
    s1 = json.load(f)
plant_names  = s1["plant_names"]
plant_to_idx = s1["plant_to_idx"]
T1 = 1.0  # temperature not calibrated yet

ckpt  = torch.load(config.STAGE1_MODEL, map_location=DEVICE)
model = models.resnet50(weights=None)
model.fc = nn.Sequential(
    nn.Dropout(0.3), nn.Linear(model.fc.in_features, len(plant_names)))
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE).eval()

tf = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

correct = total = 0
with open(f"{config.SPLITS_DIR}/test.csv") as f:
    for row in csv.DictReader(f):
        label = row["label"]
        plant = label.split(config.DISPLAY_SEP)[0] if config.DISPLAY_SEP in label else label
        if plant not in plant_to_idx:
            continue
        try:
            img = tf(Image.open(row["path"]).convert("RGB")).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                logit = model(img) / T1
                pred  = torch.softmax(logit, dim=1).argmax(dim=1).item()
            if plant_names[pred] == plant:
                correct += 1
            total += 1
        except:
            pass

print(f"\n{'='*50}")
print(f"  Stage 1 — Plant Classification")
print(f"  Test Accuracy : {correct/total:.4f}  ({correct}/{total})")
print(f"{'='*50}")
print("\n✅ Stage 1 evaluation done — proceed to Stage 2 training")

---
## 🔬 Stage 2 — Per-Plant Disease Classifiers
Trains **6 separate models**, one per plant that has multiple diseases.  
Each cell is independent — you can switch WiFi between cells safely.

| Cell | Plant | Diseases | Est. Time |
|------|-------|----------|-----------|
| 9A | Apple Tree | 4 | ~25 min |
| 9B | Cassava | 3 | ~30 min |
| 9C | Chinese Rose | 2 | ~15 min |
| 9D | Corn-Maize | 5 | ~40 min |
| 9E | Pear Tree | 3 | ~20 min |
| 9F | Tomato | 3 | ~25 min |

In [ ]:
# ============================================================
# CELL 9A — Train Stage 2: Apple Tree  (~25 min)
# ============================================================
import os, sys, mlflow
mlflow.end_run()
sys.path.insert(0, "/kaggle/working/agrocure_v4")
os.chdir("/kaggle/working/agrocure_v4")
import config_v4 as config
from train_stage2_single import train_plant_model
train_plant_model("Apple Tree", config.PLANT_DISEASES["Apple Tree"])

In [ ]:
# ============================================================
# CELL 9B — Train Stage 2: Cassava  (~30 min)
# ============================================================
import os, sys, mlflow
mlflow.end_run()
sys.path.insert(0, "/kaggle/working/agrocure_v4")
os.chdir("/kaggle/working/agrocure_v4")
import config_v4 as config
from train_stage2_single import train_plant_model
train_plant_model("Cassava", config.PLANT_DISEASES["Cassava"])

In [ ]:
# ============================================================
# CELL 9C — Train Stage 2: Chinese Rose  (~15 min)
# ============================================================
import os, sys, mlflow
mlflow.end_run()
sys.path.insert(0, "/kaggle/working/agrocure_v4")
os.chdir("/kaggle/working/agrocure_v4")
import config_v4 as config
from train_stage2_single import train_plant_model
train_plant_model("Chinese Rose", config.PLANT_DISEASES["Chinese Rose"])

In [ ]:
# ============================================================
# CELL 9D — Train Stage 2: Corn-Maize  (~40 min)
# ============================================================
import os, sys, mlflow
mlflow.end_run()
sys.path.insert(0, "/kaggle/working/agrocure_v4")
os.chdir("/kaggle/working/agrocure_v4")
import config_v4 as config
from train_stage2_single import train_plant_model
train_plant_model("Corn-Maize", config.PLANT_DISEASES["Corn-Maize"])

In [ ]:
# ============================================================
# CELL 9E — Train Stage 2: Pear Tree  (~20 min)
# ============================================================
import os, sys, mlflow
mlflow.end_run()
sys.path.insert(0, "/kaggle/working/agrocure_v4")
os.chdir("/kaggle/working/agrocure_v4")
import config_v4 as config
from train_stage2_single import train_plant_model
train_plant_model("Pear Tree", config.PLANT_DISEASES["Pear Tree"])

In [ ]:
# ============================================================
# CELL 9F — Train Stage 2: Tomato  (~25 min)
# ============================================================
import os, sys, mlflow
mlflow.end_run()
sys.path.insert(0, "/kaggle/working/agrocure_v4")
os.chdir("/kaggle/working/agrocure_v4")
import config_v4 as config
from train_stage2_single import train_plant_model
train_plant_model("Tomato", config.PLANT_DISEASES["Tomato"])

---
## 📊 Post-Training: Calibration, Evaluation, EDA

In [ ]:
# ============================================================
# CELL 10 — Calibrate All Models
# ============================================================
import runpy, os, mlflow
mlflow.end_run()
os.chdir("/kaggle/working/agrocure_v4")
runpy.run_path("06_calibrate_v4.py", run_name="__main__")

In [ ]:
# ============================================================
# CELL 11 — Full Pipeline Evaluation (Stage 1 + Stage 2)
# ============================================================
import runpy, os, mlflow
mlflow.end_run()
os.chdir("/kaggle/working/agrocure_v4")
runpy.run_path("07_evaluate_v4.py", run_name="__main__")

In [ ]:
# ============================================================
# CELL 12 — Post-Training EDA
# ============================================================
import os, json, csv, torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision import transforms, models
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.calibration import calibration_curve
import sys
sys.path.insert(0, "/kaggle/working/agrocure_v4")
import config_v4 as config
import inference_v4

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EDA_DIR = "/kaggle/working/eda_plots_v4"
os.makedirs(EDA_DIR, exist_ok=True)

pipeline = inference_v4.AgroCureV4()

all_preds, all_labels, all_confs, all_paths = [], [], [], []
with open(f"{config.SPLITS_DIR}/test.csv") as f:
    test_rows = list(csv.DictReader(f))

print(f"Running inference on {len(test_rows)} test images...")
for i, row in enumerate(test_rows):
    try:
        result = pipeline.predict(row["path"])
        all_preds.append(result["label"])
        all_labels.append(row["label"])
        all_confs.append(result["confidence"])
        all_paths.append(row["path"])
    except:
        pass
    if (i+1) % 300 == 0:
        print(f"  {i+1}/{len(test_rows)}...")

all_preds    = np.array(all_preds)
all_labels   = np.array(all_labels)
all_confs    = np.array(all_confs)
correct_mask = (all_preds == all_labels)

unique_labels = sorted(set(all_labels.tolist() + all_preds.tolist()))
label_to_idx  = {l:i for i,l in enumerate(unique_labels)}
y_true = [label_to_idx[l] for l in all_labels]
y_pred = [label_to_idx[l] for l in all_preds]

print(f"\nTop-1 Accuracy : {correct_mask.mean():.4f}")
print(f"Macro F1       : {f1_score(y_true,y_pred,average='macro',zero_division=0):.4f}")

# Per-class F1 chart
f1s   = f1_score(y_true, y_pred, average=None, zero_division=0)
precs = precision_score(y_true, y_pred, average=None, zero_division=0)
recs  = recall_score(y_true, y_pred, average=None, zero_division=0)
per_class_df = pd.DataFrame({
    "class": unique_labels, "f1": f1s,
    "precision": precs, "recall": recs
}).sort_values("f1")

fig, ax = plt.subplots(figsize=(14, 14))
colors  = ['#d32f2f' if f<0.7 else '#f57c00' if f<0.85 else '#388e3c' for f in per_class_df["f1"]]
bars    = ax.barh(per_class_df["class"], per_class_df["f1"], color=colors)
ax.axvline(0.85, color='blue', linestyle='--', label='Target (0.85)')
ax.axvline(per_class_df["f1"].mean(), color='orange', linestyle='--',
           label=f'Mean F1 ({per_class_df["f1"].mean():.3f})')
for bar, val in zip(bars, per_class_df["f1"]):
    ax.text(val+0.005, bar.get_y()+bar.get_height()/2, f"{val:.3f}", va='center', fontsize=8)
ax.set_title('Per-Class F1 — AgroCure v4', fontsize=13, fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig(f'{EDA_DIR}/per_class_f1_v4.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ F1 chart saved")

# Confidence distribution + calibration
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(all_confs[correct_mask],  bins=30, alpha=0.7, color='green', label='Correct')
axes[0].hist(all_confs[~correct_mask], bins=30, alpha=0.7, color='red',   label='Wrong')
axes[0].set_title('Confidence Distribution — v4')
axes[0].set_xlabel('Confidence'); axes[0].legend()
prob_true, prob_pred = calibration_curve(correct_mask, all_confs, n_bins=10)
axes[1].plot([0,1],[0,1],'k--',label='Perfect')
axes[1].plot(prob_pred, prob_true,'s-',color='blue',label='Model')
axes[1].set_title('Calibration Curve — v4')
axes[1].set_xlabel('Predicted Confidence'); axes[1].legend()
plt.tight_layout()
plt.savefig(f'{EDA_DIR}/confidence_v4.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Confidence plot saved")

# Top-10 wrong predictions
wrong_idx = np.where(~correct_mask)[0]
top_wrong = wrong_idx[np.argsort(all_confs[wrong_idx])[::-1][:10]]
fig, axes = plt.subplots(2, 5, figsize=(20, 9))
fig.suptitle('Top-10 High-Confidence Wrong Predictions — v4', fontsize=13, fontweight='bold')
for ax, idx in zip(axes.flatten(), top_wrong):
    try:
        img = Image.open(all_paths[idx]).convert('RGB').resize((180,180))
        ax.imshow(img)
    except:
        pass
    ax.set_title(f"True: {all_labels[idx].split(config.DISPLAY_SEP)[-1]}\n"
                 f"Pred: {all_preds[idx].split(config.DISPLAY_SEP)[-1]}\n"
                 f"Conf: {all_confs[idx]:.2f}", fontsize=8, color='red')
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{EDA_DIR}/misclassified_v4.png', dpi=120, bbox_inches='tight')
plt.show()
print("✓ Misclassified images saved")

print(f"\n{'='*50}")
print(f"  POST-TRAINING EDA COMPLETE — v4")
print(f"  Top-1 Accuracy : {correct_mask.mean():.4f}")
print(f"  Macro F1       : {f1_score(y_true,y_pred,average='macro',zero_division=0):.4f}")
print(f"{'='*50}")

In [ ]:
# ============================================================
# CELL 13 — Upload Metrics to DagsHub
# ============================================================
import json, mlflow
mlflow.end_run()

REPORT_DIR = "/kaggle/working/agrocure_v4/report"

with open(f"{REPORT_DIR}/metrics_v4.json") as f:
    metrics = json.load(f)

with mlflow.start_run(run_name="agrocure_v4_final"):
    mlflow.log_params({
        "model":        "EfficientNetV2-S x2 (hierarchical)",
        "architecture": "Stage1(plant) + Stage2(disease)",
        "num_classes":  30,
        "img_size":     224,
        "batch_size":   32,
        "lr":           1e-4,
    })
    mlflow.log_metrics({
        "test_top1_accuracy": metrics["top1_accuracy"],
        "test_macro_f1":      metrics["macro_f1"],
        "mean_conf":          metrics["mean_conf"],
    })
    print("✅ Metrics logged to DagsHub!")
    print("View → https://dagshub.com/Faira73/Agro-Mind.mlflow")

In [ ]:
# ============================================================
# CELL 14 — Save Everything
# ============================================================
import os, shutil, zipfile, json, subprocess

WORK_DIR = "/kaggle/working/agrocure_v4"
OUT_DIR  = "/kaggle/working/agrocure_v4_output"
os.makedirs(OUT_DIR, exist_ok=True)

# Model files
print("=== Model Files ===")
models_dir = f"{WORK_DIR}/models"
for fname in os.listdir(models_dir):
    src = f"{models_dir}/{fname}"
    shutil.copy(src, f"{OUT_DIR}/{fname}")
    print(f"  ✓ {fname}  ({os.path.getsize(src)/1e6:.1f} MB)")

# Report
print("\n=== Report ===")
for fname in os.listdir(f"{WORK_DIR}/report"):
    shutil.copy(f"{WORK_DIR}/report/{fname}", f"{OUT_DIR}/{fname}")
    print(f"  ✓ {fname}")

# EDA plots
print("\n=== EDA Plots ===")
eda_src = "/kaggle/working/eda_plots_v4"
for fname in os.listdir(eda_src):
    shutil.copy(f"{eda_src}/{fname}", f"{OUT_DIR}/{fname}")
    print(f"  ✓ {fname}")

# Splits zip
with zipfile.ZipFile(f"{OUT_DIR}/splits_v4.zip","w",zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(f"{WORK_DIR}/splits"):
        for file in files:
            fp = os.path.join(root, file)
            zf.write(fp, os.path.relpath(fp, WORK_DIR))
print("\n  ✓ splits_v4.zip")

# Full backup
print("\nZipping full backup...")
with zipfile.ZipFile(f"{OUT_DIR}/agrocure_v4_backup.zip","w",zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(WORK_DIR):
        dirs[:] = [d for d in dirs if d != "dataset"]
        for file in files:
            fp = os.path.join(root, file)
            zf.write(fp, os.path.relpath(fp, WORK_DIR))
size = os.path.getsize(f"{OUT_DIR}/agrocure_v4_backup.zip")/1e6
print(f"  ✓ agrocure_v4_backup.zip  ({size:.0f} MB)")

# Summary
print(f"\n{'='*55}")
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR,f))/1e6
    print(f"  {f:<50} {size:.1f} MB")
print(f"{'='*55}")
print("\n✅ Done! → Output tab → download agrocure_v4_backup.zip")